# 单圈彩色扫描 → PL测量与三维模型

按顺序运行，**不要Run All**。仍为初始化加8步，共九个代码格。只采一圈24张，不再翻转物体、拍第二圈或点选配准。

第7步由PL逐像素判定，PS估计参数并清理二值结果。第8步利用同圈原图核验投影、复核待定证据，生成外壳模型并近似着色；不把待定区域当成测得的实体，不凭空补齐。实体屏顶部显示当前步骤、耗时和PL体积估计，下方保持摄像头画面。


In [ ]:
from pathlib import Path
from datetime import datetime
import json, sys, time
import numpy as np
from PIL import Image
from IPython.display import display
from pynq import Overlay

DATA_DIR = Path('/home/xilinx/jupyter_notebooks/User/Overlays/OV5640_UI/mask_rtl_1010_v1')
COLOR_BIT = DATA_DIR / 'scan3d.bit'
RUN_DIR = DATA_DIR / ('1014_彩色原图扫描_%s_v1' % datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
RUN_DIR.mkdir(parents=True, exist_ok=False)
sys.path.insert(0, str(DATA_DIR))
import importlib, scan_pipeline
scan_pipeline = importlib.reload(scan_pipeline)
if Path(scan_pipeline.__file__).resolve() != (DATA_DIR / 'scan_pipeline.py').resolve():
    raise RuntimeError('加载了错误目录的 scan_pipeline.py')
if not hasattr(scan_pipeline, 'capture_views_demo'):
    raise RuntimeError('请先把演示版 scan_pipeline.py 更新到板上目录')
import camera_control
camera_control = importlib.reload(camera_control)
if getattr(camera_control, 'IMPLEMENTATION_VERSION', 0) != 20:
    raise RuntimeError('请上传本包camera_control.py和v20 bit/hwh')
import adaptive_scan
adaptive_scan = importlib.reload(adaptive_scan)
if getattr(adaptive_scan, 'ALGORITHM_VERSION', 0) < 5:
    raise RuntimeError('请上传本包的 adaptive_scan.py')
import rtl_mask_demo
rtl_mask_demo = importlib.reload(rtl_mask_demo)
if Path(rtl_mask_demo.__file__).resolve() != (DATA_DIR / 'rtl_mask_demo.py').resolve():
    raise RuntimeError('加载了错误目录的 rtl_mask_demo.py')
if getattr(rtl_mask_demo, 'FRAME_READER_VERSION', 0) < 29:
    raise RuntimeError('请上传v29的rtl_mask_demo.py，修正DDR四像素顺序')
from rtl_mask_demo import grab_raw, require_framebuffer_reservation
import importlib, scan_pipeline
importlib.invalidate_caches()
scan_pipeline = importlib.reload(scan_pipeline)
from scan_pipeline import (save_color_frame, calibrate_turntable_photos,
                           measure_period, capture_color_views,
                           color_difference_mask, segment_color_scan,
                           clean_object_mask)
if not COLOR_BIT.is_file() or not COLOR_BIT.with_suffix('.hwh').is_file():
    raise FileNotFoundError('找不到PL掩膜与彩照共存bit：%s；请先核对路径' % COLOR_BIT)
print('本次结果文件夹：', RUN_DIR)
require_framebuffer_reservation()
print('先移走整个转台，固定相机和灯光；本格加载彩照、PL掩膜和PL重建共用bit。')
if 'pl_device' in globals():
    pl_device.close()
import pl_mask
pl_mask = importlib.reload(pl_mask)
if getattr(pl_mask,'LOW_CONTRAST_RECOVERY_VERSION',0)<66:
    raise RuntimeError('请一起覆盖v66包里的pl_mask.py、subject_region.py和competitive_colour.py')
if getattr(pl_mask, 'FRAME_ORDER_METADATA_VERSION', 0) < 29:
    raise RuntimeError('请一起更新v29的pl_mask.py，记录原图与掩膜像素顺序')
if getattr(pl_mask, 'IMPLEMENTATION_VERSION', 0) != 20:
    raise RuntimeError('请上传本包的pl_mask.py')
if 'scan_screen' in sys.modules:
    sys.modules['scan_screen'].detach()  # stop the old UI writer before reprogramming
# HWH dividers were generated for a different boot-time PS PLL.
from pynq import Clocks as _scan98_clocks
import scan_clock as _scan98_clock
_scan98_clock = importlib.reload(_scan98_clock)
if (Path(_scan98_clock.__file__).resolve() != (DATA_DIR / 'scan_clock.py').resolve()
        or getattr(_scan98_clock, 'VERSION', 0) != 98):
    raise RuntimeError('请上传本包scan_clock.py到当前Notebook目录')
ol_color = Overlay(str(COLOR_BIT), download=False)
_scan98_plan = _scan98_clock.prepare(ol_color, _scan98_clocks)
ol_color.download()
_scan98_clock_info = _scan98_clock.verify(_scan98_clocks, _scan98_plan,
    RUN_DIR / '1028_display_clock_v98.json')
print('时钟寄存器核对：FCLK0 %.6f MHz；屏幕时钟计算值 %.6f MHz' % (
    _scan98_clock_info['actual_fclk0_register_mhz'],
    _scan98_clock_info['computed_display_clock_mhz']), flush=True)
pl_device = pl_mask.PLMaskSegmenter(ol_color)
pl_device.self_test()
time.sleep(2)
print('共用bit已加载，板上PL掩膜自检通过。运行下一格拍无转台背景。')
import scan_screen
scan_screen = importlib.reload(scan_screen)
scan_screen.attach(ol_color, RUN_DIR, COLOR_BIT.with_suffix('.hwh'))

# Read-only, frame-latched display diagnostics after the normal hardware self-test.
import screen_probe as _probe99
_probe99 = importlib.reload(_probe99)
if (Path(_probe99.__file__).resolve() != (DATA_DIR / 'screen_probe.py').resolve()
        or getattr(_probe99, 'VERSION', 0) != 99):
    raise RuntimeError('请上传本包screen_probe.py到当前Notebook目录')
_display99_status = _probe99.observe(ol_color, RUN_DIR)


## 1. 无转台背景（彩色）

把**整个转台移出镜头**。相机、灯、桌面和镜头方向都不能动。运行下一格会另存原始帧 `1010_no_turntable_v1.npy` 和可看的彩色图 `1010_no_turntable_v1.png`。画面要正常、清晰、没有条纹。
本格固定增益和白平衡，自动拍摄多档真实曝光背景。下一步放入空转台后自动选择不过曝的一档，并配对使用同档背景；之后标定与扫描固定沿用这组参数。无需手动调曝光。


In [ ]:
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
with scan_screen.step(1, globals()):
    camera_control = importlib.reload(camera_control)
    if getattr(camera_control, 'EXPOSURE_PAIR_VERSION', 0) < 24:
        raise RuntimeError('请覆盖本包 camera_control.py 后重开Notebook')
    exposure_bank = None  # Failed retries must not reuse an older scene's bank.
    camera_settings = camera_control.lock_for_scan(
        ol_color, out_path=RUN_DIR / '1016_camera_settings_v20.json')
    exposure_bank = camera_control.capture_exposure_backgrounds(
        ol_color, grab_raw, camera_settings, RUN_DIR / '1016_exposure_background_bank_v24.npz')
    time.sleep(1)
    bg565 = grab_raw()
    np.save(str(RUN_DIR / '1010_no_turntable_v1.npy'), bg565)
    save_color_frame(bg565, RUN_DIR / '1010_no_turntable_v1.png')
    display(Image.open(RUN_DIR / '1010_no_turntable_v1.png'))
    print('无转台背景已保存；看清楚后再放转台。')


## 2. 放空转台，再拍一张彩色图

转台放回刚才要测物体的位置，**台面上不能有物体或手**。转台可以先断电停住。背景与灯光保持不变。此格保存 `1010_empty_turntable_v1.npy/.png`。如果两张照片里除了转台还有人影/门窗变化，重拍。
本格会自动选择曝光，并更新对应的无转台参考；两张标定照片来自同一曝光，不使用软件调亮生成的背景。自动拍摄期间保持相机、转台和照明不动。


In [ ]:
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
with scan_screen.step(2, globals()):
    camera_control.verify_for_scan(ol_color, camera_settings)
    if not isinstance(globals().get('exposure_bank'), dict):
        raise RuntimeError('需要本次曝光背景组，请从第1步重新采集')
    bg565, _exposure_table, camera_settings = camera_control.select_exposure_pair(
        ol_color, grab_raw, exposure_bank, out_path=RUN_DIR / '1016_camera_settings_v20.json')
    # Replace the no-table reference only with the captured photo at the selected setting.
    np.save(str(RUN_DIR / '1010_no_turntable_v1.npy'), bg565)
    save_color_frame(bg565, RUN_DIR / '1010_no_turntable_v1.png')
    time.sleep(1)
    tt565, background_sigma, background_info = adaptive_scan.capture_background_reference(grab_raw)
    np.save(str(RUN_DIR / '1010_empty_turntable_v1.npy'), tt565)
    np.savez_compressed(str(RUN_DIR / '1016_background_noise_v12.npz'), sigma=background_sigma, info_json=json.dumps(background_info, ensure_ascii=False))
    save_color_frame(tt565, RUN_DIR / '1010_empty_turntable_v1.png')
    display(Image.open(RUN_DIR / '1010_empty_turntable_v1.png'))
    print('空转台原图及噪声参考已保存。')


## 3. 自动量转台的直径

程序比较两张彩色照片，抠出“新出现的转台”，但**只用其上部的稳定宽带量直径**，避开下方反光和桌面阴影。曝光变化只参考画面顶部；底边反光不会让标定直接报错。若找不到可靠边界会报错，不会悄悄用整张差分图的最宽处。打开 `1010_turntable_calibration_v1.png`：黄色两点应是转台左右边缘，蓝线应靠近物体放上去后的接触面，红线应穿过转轴。另存的 `_difference.png` 可用来排查背景变化。`GROUND_ROW_OVERRIDE` 只在蓝线不准时填一个实际像素行，重新运行本格；无需重新拍照片。

In [ ]:
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
with scan_screen.step(3, globals()):
    # 标定加载修正 v23：直接读实际源码，绕过旧模块和函数缓存
    _cal_path = DATA_DIR / 'scan_pipeline.py'
    _cal_live = {'__file__': str(_cal_path), '__name__': '_scan_calibration_live'}
    _cal_source = _cal_path.read_text(encoding='utf-8')
    exec(compile(_cal_source, str(_cal_path), 'exec'), _cal_live)
    calibrate_turntable_photos = _cal_live['calibrate_turntable_photos']
    _cal_version = _cal_live.get('CALIBRATION_BOUNDARY_VERSION', 0)
    if _cal_version < 22 or '转台贴到画面上下边缘，请调整构图' in calibrate_turntable_photos.__code__.co_consts:
        raise RuntimeError('磁盘上的标定代码仍旧（版本%s）；请覆盖这个确切文件：%s' % (_cal_version, _cal_path))
    print('第3步：输入核对 v96；标定算法 v%s' % _cal_version, flush=True)
    _cal_verified_camera = camera_control.verify_for_scan(ol_color, camera_settings)
    TURNTABLE_MM = 200.0  # 你的转台实际直径：20 cm = 200 mm
    GROUND_ROW_OVERRIDE = None  # 蓝线不准时才填台面像素行，例如 305
    # Use the actual saved pair from this RUN_DIR, not overwriteable notebook variables.
    import hashlib as _cal_hashlib, uuid as _cal_uuid, platform as _cal_platform
    _cal_bg_path = RUN_DIR / '1010_no_turntable_v1.npy'
    _cal_tt_path = RUN_DIR / '1010_empty_turntable_v1.npy'
    _cal_bg565 = np.load(str(_cal_bg_path), allow_pickle=False)
    _cal_tt565 = np.load(str(_cal_tt_path), allow_pickle=False)
    if (_cal_bg565.shape != (480, 800) or _cal_tt565.shape != (480, 800)
            or _cal_bg565.dtype != np.uint16 or _cal_tt565.dtype != np.uint16):
        raise ValueError('本次目录的标定原图必须为480x800 uint16；未改用其他目录的图片')
    _cal_input_record = dict(version=96, run_dir=str(RUN_DIR.resolve()),
        source_path=str(_cal_path.resolve()),
        source_sha256=_cal_hashlib.sha256(_cal_source.encode('utf-8')).hexdigest(),
        background_path=str(_cal_bg_path.resolve()), turntable_path=str(_cal_tt_path.resolve()),
        background_array_sha256=_cal_hashlib.sha256(_cal_bg565.tobytes()).hexdigest(),
        turntable_array_sha256=_cal_hashlib.sha256(_cal_tt565.tobytes()).hexdigest(),
        memory_background_matches_saved=bool(np.array_equal(globals().get('bg565'), _cal_bg565)),
        memory_turntable_matches_saved=bool(np.array_equal(globals().get('tt565'), _cal_tt565)),
        diameter_mm=float(TURNTABLE_MM), ground_row_override=GROUND_ROW_OVERRIDE,
        numpy_version=np.__version__, python_version=_cal_platform.python_version(),
        camera_settings=camera_settings, verified_camera=_cal_verified_camera,
        algorithm_version=_cal_version, input_source='saved_pair_in_current_run_directory',
        other_scan_used=False)
    print('本次标定目录：', RUN_DIR, flush=True)
    if not (_cal_input_record['memory_background_matches_saved'] and
            _cal_input_record['memory_turntable_matches_saved']):
        print('内存图片与本次存档不同；本格使用上述目录中的两张原图。', flush=True)
        # Retain both pairs before rebinding. Do not lose the original RAM evidence.
        _cal_difference_path = RUN_DIR / ('1028_calibration_input_difference_%s_v96.npz' % _cal_uuid.uuid4().hex)
        _cal_difference_arrays = dict(background565=_cal_bg565, turntable565=_cal_tt565)
        for _cal_name, _cal_global in (('memory_background565', 'bg565'), ('memory_turntable565', 'tt565')):
            _cal_previous = globals().get(_cal_global)
            if isinstance(_cal_previous, np.ndarray) and _cal_previous.dtype == np.uint16 and _cal_previous.shape == (480, 800):
                _cal_difference_arrays[_cal_name] = _cal_previous.copy()
        try:
            np.savez_compressed(str(_cal_difference_path),
                metadata_json=np.asarray(json.dumps(_cal_input_record, ensure_ascii=False, default=str)),
                actual_source_text=np.asarray(_cal_source), **_cal_difference_arrays)
            print('内存与存档差异证据：', _cal_difference_path, flush=True)
        except Exception as _cal_difference_error:
            print('输入差异证据未保存：%s' % _cal_difference_error, flush=True)
        display(Image.fromarray(_cal_live['rgb565_to_rgb'](_cal_bg565)))
        display(Image.fromarray(_cal_live['rgb565_to_rgb'](_cal_tt565)))
    # Keep the subsequent mask reference consistent with this exact pair.
    bg565, tt565 = _cal_bg565, _cal_tt565
    try:
        cal = calibrate_turntable_photos(
            _cal_bg565, _cal_tt565, diameter_mm=TURNTABLE_MM,
            preview_path=RUN_DIR / '1010_turntable_calibration_v1.png',
            ground_row_override=GROUND_ROW_OVERRIDE)
    except Exception as _cal_failure:
        _cal_input_record.update(status='rejected', error_type=type(_cal_failure).__name__, error=str(_cal_failure))
        _cal_failure_path = RUN_DIR / ('1028_calibration_failure_%s_v96.npz' % _cal_uuid.uuid4().hex)
        try:
            np.savez_compressed(str(_cal_failure_path),
                background565=_cal_bg565, turntable565=_cal_tt565,
                metadata_json=np.asarray(json.dumps(_cal_input_record, ensure_ascii=False, default=str)),
                actual_source_text=np.asarray(_cal_source))
            print('本次失败输入、代码和参数：', _cal_failure_path, flush=True)
        except Exception as _cal_save_error:
            print('标定失败证据未保存：%s' % _cal_save_error, flush=True)
        raise
    _cal_input_record['status'] = 'accepted'
    cal['calibration_input_evidence'] = _cal_input_record
    cal['camera_settings'] = camera_settings
    with open(RUN_DIR / '1010_calibration_v1.json', 'w', encoding='utf-8') as f:
        json.dump(cal, f, ensure_ascii=False, indent=2)
    display(Image.open(RUN_DIR / '1010_turntable_calibration_v1.png'))
    print('直径=%.1f px；比例=%.4f mm/px；转轴列=%.1f；台面行=%.1f' %
          (cal['width_px'], cal['mm_per_px'], cal['axis_col'], cal['v_ground']))
    print('差分诊断图：', RUN_DIR / '1010_turntable_calibration_v1_difference.png')
    print('请人工核对黄/蓝/红线；不准就不要继续。')


## 4. 放物体，检查彩色原图与初始掩膜

保持当前共用bit、相机、灯光和空转台背景不变，把物体放在台面中央。下一格拍一张真实彩色图，将照片和空台参考送入PL生成掩膜，并展示二值清理结果。若照片冻结、主体缺失、转台被识别成物体，就先调整现场；不要继续采集。


In [ ]:
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
with scan_screen.step(4, globals()):
    scan = None  # 新预览/采集/分割开始时不沿用旧重建输入。
    import importlib, pl_mask, background_model
    importlib.invalidate_caches()
    background_model = importlib.reload(background_model)
    pl_mask = importlib.reload(pl_mask)
    if getattr(pl_mask,'LOW_CONTRAST_RECOVERY_VERSION',0)<76:
        raise RuntimeError('请覆盖v76包里的pl_mask.py、subject_region.py、subject_evidence.py和competitive_colour.py')
    if getattr(pl_mask, 'BACKGROUND_MODEL_VERSION', 0) < 26 or background_model.IMPLEMENTATION_VERSION < 26:
        raise RuntimeError('请一起覆盖 pl_mask.py 和 background_model.py')
    if getattr(pl_device, 'REFERENCE_GROUP_DRIVER_VERSION', 0) < 25:
        pl_device.close()
        pl_device = pl_mask.PLMaskSegmenter(ol_color)
        pl_device.self_test()
    camera_control.verify_for_scan(ol_color, camera_settings)
    preview565 = grab_raw().copy()
    save_color_frame(preview565, RUN_DIR / '1014_target_color_preview_v1.png')
    preview = pl_mask.segment_frames(pl_device, [preview565], tt565, cal['v_ground'],
                                     background_sigma, directory=RUN_DIR, axis_col=cal['axis_col'], calibration=cal)
    preview_clean = preview['masks'][0]
    Image.fromarray(255 - preview_clean * 255).save(RUN_DIR / '1014_target_mask_preview_v1.png')
    print('PL像素与区域前景判定已完成（v76）；请对照原图检查黑/橙区域。')
    from PIL import ImageDraw
    measurement_preview = Image.open(RUN_DIR / '1014_target_color_preview_v1.png').copy()
    wx0,wy0,wx1,wy1=map(int,preview['measurement_window_xyxy'])
    ImageDraw.Draw(measurement_preview).rectangle((wx0,wy0,wx1-1,wy1-1),outline='red')
    print('请让物体转动整圈时都留在红框内；框外变化仅用于背景参数估计。')
    display(measurement_preview)
    mask_view=np.repeat((255-preview_clean*255)[:,:,None],3,axis=2)
    interior_uncertain=(preview['interior_unknown_masks'][0]!=0)|(preview['interior_inferred_masks'][0]!=0)
    mask_view[interior_uncertain]=[255,170,0]
    display(Image.fromarray(mask_view))
    print('黑=PL及区域证据支持的主体，橙=仍待定，白=当前背景判定；请先检查原图物体是否进入黑色掩膜。')
    np.savez_compressed(str(RUN_DIR/'1022_preview_evidence_v53.npz'),
        frame565=preview565,reference565=tt565,ground_row=cal['v_ground'],
        calibration_json=json.dumps(cal,ensure_ascii=False),
        reference_sigma_available=background_sigma is not None,
        reference_sigma=background_sigma if background_sigma is not None else np.zeros((480,800,3),np.float32),**preview)


## 5. 黑标记自动测转速

胶带和唯一黑标记应在采空台参考前贴好；若刚加胶带，先重新完成原第2、3步。让转台保持连续匀速，运行本格自动观察标记多次经过同一位置，正常无需按回车计时。测速区域在标定台面下方，不使用被测物体的纹理。

自动测量不可靠时，本格可输入r重试、m使用原手动计时或q取消；不要重复点击运行。默认支持约2～90秒/圈，最多观察5分钟，可按Notebook中断按钮停止。平均周期不能替代角度编码器，也不能保证扫描期间转速不变。


In [ ]:
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
with scan_screen.step(5, globals()):
    period_s = None
    speed_info = None
    import marker_speed
    marker_speed = importlib.reload(marker_speed)
    if Path(marker_speed.__file__).resolve() != (DATA_DIR / 'marker_speed.py').resolve() or marker_speed.VERSION < 78:
        raise RuntimeError('请上传本包marker_speed.py到同目录')
    if not isinstance(globals().get('cal'), dict):
        raise RuntimeError('先完成第3步转台标定，再自动测速')
    camera_control.verify_for_scan(ol_color, camera_settings)
    period_s, speed_info = marker_speed.measure_period_demo(
        grab_raw, cal, RUN_DIR, ask=scan_screen.ask, manual_measure=measure_period)
    if period_s is None:
        print('已取消测转速；下一步不能运行。')
    else:
        print('采用 %.3f 秒/圈；24 视角间隔约 %.3f 秒' % (period_s, period_s / 24.0))


## 6. 单圈采集24张彩色原图

保持物体摆稳、转台匀速，运行下一格即开始采集一圈24张及首尾核对帧。相机、转台位置、灯光和已锁定的相机参数保持不变。原图和角度保存到本次目录 `1014_color24_v1.npz`，预览为 `1014_color_sheet_v1.png`。不需要翻转物体。


In [ ]:
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
with scan_screen.step(6, globals()):
    scan = None
    color_scan = None
    if period_s is None:
        raise RuntimeError('先完成测转速；若正在等待输入，不要再次点击运行')
    importlib.invalidate_caches()
    scan_pipeline = importlib.reload(scan_pipeline)
    if Path(scan_pipeline.__file__).resolve() != (DATA_DIR / 'scan_pipeline.py').resolve():
        raise RuntimeError('加载的 scan_pipeline.py 路径不对')
    if getattr(scan_pipeline, 'SINGLE_CIRCLE_FLOW_VERSION', 0) < 71:
        raise RuntimeError('请覆盖v71包的scan_pipeline.py，再重跑本格')
    if isinstance(globals().get('speed_info'), dict):
        marker_speed.validate_for_capture(speed_info, period_s, cal)
    camera_control.verify_for_scan(ol_color, camera_settings)
    color_scan = scan_pipeline.capture_color_views_demo(
        grab_raw, period_s, RUN_DIR, n_views=24)
    if isinstance(globals().get('speed_info'), dict):
        marker_speed.attach_to_capture(color_scan, speed_info, RUN_DIR, calibration=cal)
    print('已保存单圈24张彩色原图：', RUN_DIR / '1014_color24_v1.npz')
    display(Image.open(RUN_DIR / '1014_color_sheet_v1.png'))


## 7. PL生成掩膜并检查

处理第6步已保存的单圈原图，不重新拍摄。原图和空台参考通过DMA送入PL逐像素判定，PS仅负责参数估计、二值清理和证据复核。保存正式掩膜、待定区域及首尾诊断；首尾不一致时先查看本次 `closure_comparison.png`。


In [ ]:
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
with scan_screen.step(7, globals()):
    scan = None  # 新预览/采集/分割开始时不沿用旧重建输入。
    if str(color_scan.get('frame_pixel_order', '')) != rtl_mask_demo.FRAME_PIXEL_ORDER or \
            str(cal.get('frame_pixel_order', '')) != rtl_mask_demo.FRAME_PIXEL_ORDER:
        raise RuntimeError('这组采集/标定仍是旧像素顺序；请用新版原流程从标定到扫描重新运行，不能混用新旧数据')
    import importlib, pl_mask, background_model
    importlib.invalidate_caches()
    background_model = importlib.reload(background_model)
    pl_mask = importlib.reload(pl_mask)
    if getattr(pl_mask,'LOW_CONTRAST_RECOVERY_VERSION',0)<76:
        raise RuntimeError('请覆盖v76包里的pl_mask.py、subject_region.py、subject_evidence.py和competitive_colour.py')
    if getattr(pl_mask, 'BACKGROUND_MODEL_VERSION', 0) < 26 or background_model.IMPLEMENTATION_VERSION < 26:
        raise RuntimeError('请一起覆盖 pl_mask.py 和 background_model.py')
    if getattr(pl_device, 'REFERENCE_GROUP_DRIVER_VERSION', 0) < 25:
        pl_device.close()
        pl_device = pl_mask.PLMaskSegmenter(ol_color)
        pl_device.self_test()
    import importlib, scan_pipeline, pl_mask
    importlib.invalidate_caches()
    scan_pipeline = importlib.reload(scan_pipeline)
    pl_mask = importlib.reload(pl_mask)
    if getattr(pl_mask,'LOW_CONTRAST_RECOVERY_VERSION',0)<76:
        raise RuntimeError('请覆盖v76包里的pl_mask.py、subject_region.py、subject_evidence.py和competitive_colour.py')
    if getattr(scan_pipeline, 'COLOR_PHOTO_REPAIR_VERSION', 0) < 5:
        raise RuntimeError('板上的 scan_pipeline.py 是旧版；先上传本包的同名文件')
    if getattr(pl_mask, 'FAILURE_EVIDENCE_VERSION', 0) < 24:
        raise RuntimeError('请覆盖本包 pl_mask.py')
    scan = pl_mask.segment_scan(
        pl_device, color_scan, tt565, cal['v_ground'],
        out_path=RUN_DIR / '1014_scan24_color_masks_v1.npz',
        preview_path=RUN_DIR / '1014_color_mask_sheet_v1.png',
        reference_sigma=background_sigma, axis_col=cal['axis_col'], calibration=cal)
    scan['color_frames565'] = color_scan['frames565']
    scan['diagnostic_archive_path'] = str((RUN_DIR / '1014_scan24_color_masks_v1.npz').resolve())
    print('正式掩膜已保存；首尾IoU %.3f' % scan['closure_iou'])
    display(Image.open(RUN_DIR / '1014_color_mask_sheet_v1.png'))


## 8. 单圈PL测量与彩色模型

复用同圈照片、掩膜和标定，不重新加载bit或采背景。先核对PL投影与掩膜，再由PS细化外形、用同圈照片近似着色。仅在已保存文件逐项一致时释放重复诊断数组，减少板上内存占用；原图、严格掩膜和磁盘证据保留。

只打开标出的主模型。PL体积为视觉外壳估计，橙色待定区域不代表确定材料。单圈照片不能保证恢复台面遮挡处、底面及不可见凹陷，形状仍需核对。


In [ ]:
from datetime import datetime
import importlib, scan_screen
if getattr(scan_screen, 'VERSION', 0) < 88:
    scan_screen.detach()
    scan_screen = importlib.reload(scan_screen)
# Stop the old viewer before refreshing its gesture code; keep scan data and the loaded bit.
scan_screen.detach()
with scan_screen.step(8, globals()):
    if not isinstance(globals().get('scan'), dict):
        print('请先完成第7步正式掩膜检查；当前没有可用的整圈输入。')
    else:
        if str(cal.get('frame_pixel_order', '')) != rtl_mask_demo.FRAME_PIXEL_ORDER:
            raise RuntimeError('标定来自旧DDR像素顺序；请用新版原流程重新标定并扫描，不直接重跑第8步')
        if str(scan.get('frame_pixel_order', '')) != rtl_mask_demo.FRAME_PIXEL_ORDER:
            raise RuntimeError('掩膜仍来自旧DDR像素顺序；请重新完成第7步，不混用新旧数据')
        if scan['closure_iou'] < 0.8:
            raise RuntimeError('首尾 IoU=%.3f，未通过0.8检查；查看本次目录 closure_comparison.png，先核对首尾原图及掩膜' % scan['closure_iou'])
        import importlib, scan_pipeline, ps_demo
        importlib.invalidate_caches()
        ps_demo = importlib.reload(ps_demo)
        if getattr(ps_demo, 'PIXEL_CENTER_PROJECTION_VERSION', 0) < 1:
            raise RuntimeError('请同时上传本包的 ps_demo.py，统一像素中心坐标')
        scan_pipeline = importlib.reload(scan_pipeline)
        if Path(scan_pipeline.__file__).resolve() != (DATA_DIR / 'scan_pipeline.py').resolve():
            raise RuntimeError('加载的 scan_pipeline.py 路径不对：%s' % scan_pipeline.__file__)
        if getattr(scan_pipeline, 'COLOR_PHOTO_REPAIR_VERSION', 0) < 5:
            raise RuntimeError('板上的 scan_pipeline.py 是旧版；上传本包的同名文件')
        if getattr(scan_pipeline, 'GEOMETRY_ALIGNMENT_VERSION', 0) < 5:
            raise RuntimeError('板上的 scan_pipeline.py 尚未更新三维模型对齐；上传本包的同名文件')
        pl_device._check_overlay()
        ol_mask = ol_color  # 同一个bit已有recon_0，不再切换硬件或重新采背景
        result_dir = RUN_DIR / ('1014_photo_guided_result_%s_v1' % datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        print('正在重建，请稍候……', flush=True)
        import scan_runtime
        scan_runtime = importlib.reload(scan_runtime)
        if Path(scan_runtime.__file__).resolve().parent != DATA_DIR.resolve():
            raise RuntimeError('scan_runtime.py必须与scan_pipeline.py同目录')
        archive = scan.get('diagnostic_archive_path') or RUN_DIR / '1014_scan24_color_masks_v1.npz'
        memory_info = scan_runtime.compact(scan, archive)
        report, notes = scan_pipeline.reconstruct_demo(scan, cal, ol_mask, result_dir)
        report['capture_mode'] = 'single_circle_24_views'
        report['runtime_memory_optimization'] = memory_info
        with (result_dir / 'result.json').open('w', encoding='utf8') as stream:
            json.dump(report, stream, ensure_ascii=False, indent=2)
        print('PL 视觉外壳体积估计 %.2f cm³；形状仍需核对' % report['volume_cm3'])
        if report.get('interior_evidence',{}).get('unknown_pixels',0):
            print('三维模型含内部待定区域的保守外壳；不代表测得了缺口深度。')
        if report.get('detail_mesh_file'):
            print('【打开这个三维模型】', result_dir / report['detail_mesh_file'])
        elif report.get('detail_surface_file'):
            print('【三维模型备用点云】', result_dir / report['detail_surface_file'])
        else:
            print('【三维模型备用点云】', result_dir / report.get('pl_comparison_file', 'recon.ply'))
        print('其他 PLY 为对照数据；照片颜色只表示近似表面外观')
        if notes:
            print('需检查诊断：%s；详见 result.json 和 run_log.txt' % '、'.join(notes))


**精度边界**：普通镜头的透视、计时估角、单DDR帧缓存和照片清晰度仍会影响结果。本版修正代码一致性问题，不对某次实验的光照、缺口或颜色进行经验补偿，也不能补出未拍到的表面深度。
